# __Lecture 08: Finding patterns in biological data__

----
### **_Be sure to pull the latest notebook!!_**

#### __Announcement:__ Homework 3 is due October 27. Homework 4 will be assigned October 29.

#### __Last week, you talked about:__
- Python! (e.g. data types, lists, dictionaries, for loops, if statements, functions)

#### __Today we will:__
- warm up by writing a function that uses a dictionary
- talk about why __searching for patterns__ comes up constantly in computational biology
- learn to __read__ regular expressions using the `re` module
- __write tests__ for pattern-matching code, and use them to catch a bug

-----
# __Warm-up: defining a function__

Everything we will do today builds on the Python from lectures 6 and 7: dictionaries, `for` loops, `if`
statements, and functions with default arguments. 

#### __In groups (5 minutes):__ write a function that uses a dictionary to count how many times each base occurs.

Be sure to account for sequences that contain uppercase and lowercase bases. I've provided an empty
function here (with the docstring already written). Use the test cell below to check your work.

__A note on docstrings:__ There are several styles of docstrings. I'll be writing functions with [Google-style docstrings](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings),
like the one below: a one-line summary, then `Args:`, then `Returns:`. Writing the docstring
*before* the code is a good way to force yourself to decide what the function is supposed to do.

In [ ]:
def count_bases(seq, case="upper"):
    """Count the number of times each base occurs in a sequence.

    Args:
        seq: DNA sequence, as a string.
        case: Case in which to count the bases, either "upper" or "lower".
            Defaults to "upper".

    Returns:
        A dict keyed by each nucleotide, where the value is the number of times
        that nucleotide is observed in the sequence.

    Note:
        <object>.upper() will convert the <object> string to uppercase.
    """

    assert case in ["lower", "upper"], "case needs to be either lower or upper"

    # YOUR CODE HERE

Notice that the third line below doesn't print anything. Instead, it *stops*, because of
the `assert` statement in the function. We will come back to `assert` in a big way later today.

In [ ]:
print(count_bases("AATXXGGCT"))
print(count_bases("aatTGGcT", case="lower"))

count_bases("ATGCG", case="unicorn")

----
# __Part 1: Why search for patterns?__

As we start working with sequencing data, we will want to be able to search sequences for patterns
of interest.

More broadly: a surprising amount of computational biology is __text parsing__. Here are a few examples you might run into:

In [ ]:
# 1. An influenza strain name downloaded from a sequence database.
#    Q: what year was it collected? What subtype is it?
strain = "A/New York/3/1994 (H3N2)"

# 2. A sequencing read containing a barcode (lower case) flanked by a constant sequence on each side.
#    Q: what is the barcode?
barcode = "CTAGCatcgatGATCA"

# 3. FASTQ file names from a shared directory, named by three different people.
#    Q: which gene and which replicate is each file?
filenames = [
    "Sample_01_HA_rep1_R1.fastq.gz",
    "sample-02-NA-rep2_R1.fastq.gz",
    "S03.HA.rep1.R1.fastq.gz",
]

print(strain)
print(barcode)
print(filenames)

Last week, we learned that we can ask whether one string contains another with the `in` keyword,
and that we can break a string apart with `.split()`. Both are useful, and both run out of road
quickly:

In [ ]:
# `in` can confirm a subtype we already know about...
print("H3N2" in strain)

# ...but it can't tell us the subtype of a strain we haven't seen before
mystery_strain = "A/swine/Iowa/15/1930 (H1N1)"
print(
    "H3N2" in mystery_strain
)  # False -- but there IS a subtype here, we just don't know it yet

# and .split() only helps when the text is perfectly consistent
print("Sample_01_HA_rep1_R1.fastq.gz".split("_"))
print("sample-02-NA-rep2_R1.fastq.gz".split("_"))

To use `in`, you have to already know the exact text you are looking for. The questions above are
different: we want __whatever is in that position__, we want __whatever has that shape__.

----
# __Part 2: What does it mean to search for a pattern "algorithmically"?__

Before we reach for a new module, it is worth seeing that we could do this ourselves with the Python
we already have. Say we expect sequences that look like `CTAGC`-`NNNNNN`-`GATCA`, where the six `N`s
are a barcode. (This comes up constantly: in single-cell RNA-seq a barcode tells you which cell a
read came from; in a deep mutational scanning experiment it tells you which variant.)

To find the barcode, we have to do three things:

1. __scan__: step along the sequence, one position at a time
2. __test__: at this position, does the text match what we expect?
3. __report__: if it does, pull out the piece we care about

In [ ]:
def find_barcode_by_hand(seq, upstream="CTAGC", downstream="GATCA", bclen=6):
    """Find a barcode flanked by fixed sequences, by checking every position.

    Args:
        seq: DNA sequence, as a string.
        upstream: Sequence expected immediately upstream of the barcode.
        downstream: Sequence expected immediately downstream of the barcode.
        bclen: Length of the barcode.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected design.
    """
    seq = seq.upper()
    construct_len = len(upstream) + bclen + len(downstream)

    # SCAN: try every position where the whole construct could still fit
    for i in range(len(seq) - construct_len + 1):
        # TEST: is the upstream sequence here?
        if seq[i : i + len(upstream)] != upstream:
            continue

        bc_start = i + len(upstream)
        bc_end = bc_start + bclen

        # TEST: is the downstream sequence exactly bclen bases later?
        if seq[bc_end : bc_end + len(downstream)] != downstream:
            continue

        # REPORT
        return seq[bc_start:bc_end]

    return None


print(find_barcode_by_hand("CTAGCatcgatGATCA"))
print(find_barcode_by_hand("CCAGCatagcaGATCA"))

That works! So why do we need anything else?

Because the __scan__ step is always the same ("walk along the string and try each position"), but the
__test__ step changes every time the experiment does. Suppose the upstream sequence could be `CTAGC`
*or* `CTGGC`, the barcode could be 6 *to* 8 bases long, and we only want to accept `A`, `C`, `G`, `T`
and `N` in the barcode. All of that is doable with more loops and more `if` statements, but it gets
long and hard to change.

A __regular expression__ is a compact way of writing down just the __test__: you describe the shape
you are looking for, and the `re` module does the scanning and extracting for you.

---
# __Part 3: Reading regular expressions__

Python has an extensive collection of built-in __modules__ containing special purpose functions and
objects - we have already used `math`, and on Thursday we will use `biopython`. Today's is `re`.
([The full list of built-in modules is here](https://docs.python.org/3/library/).)

#### A __regular expression__ is a sequence of characters that forms a __search pattern__.

The `re` module compiles that pattern into a matching engine and runs it against your text. The
workflow is almost always the same three steps: __compile__, __search__, __use the result__.

In [ ]:
import re

# This is an example string
seq = "ATTCCGAAA"

# 1. COMPILE the search pattern (note the r before the quotes -- more on that below)
search_pattern = re.compile(r"TC+G")

# 2. SEARCH the sequence with it
match = search_pattern.search(seq)

# 3. USE the result. A match object knows WHAT matched and WHERE.
print(match)
print("matched text:", match.group())
print("position:", match.start(), "to", match.end())

#### When there is nothing to find, you get `None` back - not an error

This is the single most common source of confusion with `re`, so it is worth seeing explicitly.
It is why the `if match is None:` check shows up in nearly every piece of regex code you will
write (or read).

In [ ]:
no_match = search_pattern.search("TTTTTT")

print(no_match)
print(no_match is None)

# this is why you ALWAYS check before calling .group()
# no_match.group()   # <- uncomment to see the error you will definitely hit at some point

__A note on quotes:__ you will see search patterns written as _raw strings_, like `r"\d{4}"`. The `r`
tells Python to leave the backslashes alone instead of interpreting them (`"\n"`, for example, is a
newline rather than a backslash and an `n`). Regular expressions use backslashes constantly, so
__always write your patterns as `r"..."`__ - it costs nothing and avoids a confusing class of bug.

## Pattern cheat sheet

You do __not__ need to memorize this. Keep it open while you read patterns - that is what it is for.

#### Matching specific characters

| pattern | matches |
| --- | --- |
| `A`, `7`, `-` | that exact character |
| `\.` `\$` `\(` `\)` `\*` `\+` | a *literal* `.` `$` `(` `)` `*` `+` - characters that have a special meaning below must be escaped with `\` |

#### Matching a class of characters

| pattern | matches |
| --- | --- |
| `.` | any single character |
| `\d` | any digit, `0`-`9` |
| `\s` | any whitespace character |
| `[ACGTN]` | any one of `A`, `C`, `G`, `T`, `N` (i.e. any one nucleotide) |
| `[a-n]` | any one lower-case letter from `a` to `n` |

#### How many times

| pattern | matches |
| --- | --- |
| `a+` | one or more `a` |
| `a{6}` | exactly six `a` |
| `a{6,8}` | between six and eight `a` |

#### Where in the string

| pattern | matches |
| --- | --- |
| `^A` | an `A` at the *beginning* of the string |
| `A$` | an `A` at the *end* of the string |

#### Grouping and naming

| pattern | meaning |
| --- | --- |
| `(...)` | group this part of the pattern together, and capture what it matched |
| `(?P<name>...)` | capture it *and* give it a name, so you can ask for it by name later |

There are many more special codes (see [the full docs](https://docs.python.org/3/library/re.html)),
and [regex101.com](https://regex101.com) will show you, piece by piece, what a pattern is doing.

## A worked example: reading a pattern one piece at a time

The trick to reading a regular expression is to go left to right and translate each piece into
English, using the tables above. Here is the influenza strain name from Part 1, and a pattern that
pulls the collection year out of it.

In [ ]:
strain = "A/New York/3/1994 (H3N2)"

# r"/(?P<year>\d{4})" reads as:
#   /           a literal forward slash
#   (?P<year>   start a captured group, and call it "year"
#   \d{4}       exactly four digits
#   )           end the group

year_pattern = re.compile(r"/(?P<year>\d{4})")

match = year_pattern.search(strain)

print(match)
print("year:", match.group("year"))

#### __Groups__ and __names__

Two of those pieces are worth calling out, because they are how you get *part* of a match back out:

* **Grouping**: `(...)` treats everything inside the parentheses as a single unit, and __captures__
  what it matched so you can ask for it separately from the rest of the pattern.
* **Naming**: `(?P<name>...)` captures it *and* labels it, so you can ask for it by name later with
  `match.group("name")` instead of counting parentheses.

Without the group, `year_pattern.search(strain).group()` would give us `"/1994"` - the slash
included - because the whole pattern matched. The group lets us say "match the slash, but only
*give me back* the four digits."

----
### __Practice 1 (in groups, ~10 minutes): read these patterns__

For each cell below:

1. __Before running it__, write down in the comment what shape the pattern describes, in plain
   English, and what you predict will be printed.
2. Then run the cell and compare.

Getting a prediction wrong here is the entire point - that is where you find out how the engine
actually behaves, rather than how you assumed it behaves.

In [ ]:
# (a) What shape does this describe? What will it print?
# your answer:

subtype_pattern = re.compile(r"\((?P<subtype>[A-Z]\d+[A-Z]\d+)\)")

for s in [
    "A/New York/3/1994 (H3N2)",
    "A/swine/Iowa/15/1930 (H1N1)",
    "B/Victoria/2/1987",
]:
    match = subtype_pattern.search(s)
    print(s, "->", match.group("subtype") if match else "no subtype found")

In [ ]:
# (b) Same pattern, with and without the ^ anchor. Which of these three find a match, and why?
# your answer:

loose = re.compile(r"ATG")
anchored = re.compile(r"^ATG")

print(loose.search("GGATGCC"))
print(anchored.search("GGATGCC"))
print(anchored.search("ATGCCGG"))

In [ ]:
# (c) Why does the gene name come out too long?
# your answer:

greedy = re.compile(r"Sample_\d+_(?P<gene>.+)_")

print(greedy.search("Sample_01_HA_rep1_R1.fastq.gz").group("gene"))

----
### __Practice 2 (in groups, ~5 minutes): now write two of your own__

Reading is most of the job, but you should write a couple from scratch so that you can practice with the syntax.
Both of these pull a __barcode__ out of a read - exactly the job we will be doing for the rest of
today and on Thursday. They are short; use the cheat sheet.

__(a)__ On Thursday, the real reads have a __16-nucleotide barcode__ sitting immediately after a
fixed adapter sequence, `AGGCGGCCGC`. Write a pattern that pulls that barcode out, named `barcode`.
Only `A`, `C`, `G`, `T` and `N` should be allowed in it - so the second read below, whose barcode
contains a `Y`, should __not__ match.

__(b)__ In a single-cell RNA-seq read, the first 16 bases are a __cell barcode__ and the next 12 are
a __UMI__, both at the very start of the read. Write one pattern that pulls out both, named `cell`
and `umi`. You will need an anchor so that it can only match at the beginning.

In [ ]:
# (a) a 16-nt barcode immediately after the fixed adapter AGGCGGCCGC
reads = [
    "CTGACAGGCGGCCGCACGTACGTACGTACGTGGG",  # barcode ACGTACGTACGTACGT
    "CTGACAGGCGGCCGCACGTACGTACGTACGYGGG",  # barcode contains a Y -- should NOT match
    "CTGACTTTTTTTTTTACGTACGTACGTACGTGGG",  # no adapter -- should NOT match
]

bc_pattern = re.compile(r"")  # <-- your pattern here

for read in reads:
    match = bc_pattern.search(read)
    print(read, "->", match.group("barcode") if match else "no match")

In [ ]:
# (b) a 16-nt cell barcode followed by a 12-nt UMI, at the very start of the read
sc_read = "ACGTACGTACGTACGTTTTTGGGGCCCCAAAAAAAA"

sc_pattern = re.compile(r"")  # <-- your pattern here

match = sc_pattern.search(sc_read)
print(match.groupdict())

#### Three designs, one shape

You have now seen three barcode designs: the `CTAGC`-`NNNNNN`-`GATCA` construct from Part 2,
Thursday's adapter-plus-16mer, and the single-cell cell-barcode-plus-UMI. They look different, but
every one of them is built from the same three ingredients:

1. something __fixed__ to anchor on (an adapter, a restriction site, the start of the read)
2. a barcode with a known __length__ and a known __alphabet__
3. sometimes, something __fixed__ on the other side

That is why a single pattern shape - `FIXED(?P<barcode>[ACGTN]{n})FIXED` - keeps reappearing.

For the rest of today we will go back to the smallest of the three, the `CTAGC`-`NNNNNN`-`GATCA`
construct, because we are going to take a pattern for it apart and find out whether it is actually
right.

----
# __Part 4: Checking that a search pattern is right__

It is easy to write an incorrect regular expression that looks like a correct one. For example, it could:
- match things it should not
- miss things it should catch

Neither failure crashes, and neither prints a warning. The pattern just quietly returns answers, and
those answers end up in your counts and your figures.

__It is a valuable skill to be able to say what the pattern should do, and then check that it does.__
That is what this section is about, and we will do it in three moves: write down the requirements,
turn them into test cases, and then find out what the requirements missed.

#### Step 1: write down the requirements

Before testing anything (and before writing the pattern, ideally) we have to state the design
precisely. Our construct is `CTAGC`-`NNNNNN`-`GATCA`, which means a valid read must satisfy all of:

__Requirements:__
1. the read __starts__ with the fixed upstream sequence `CTAGC` 
2. the barcode is exactly __6__ bases long 
3. every barcode base is one of `A`, `C`, `G`, `T`, `N` - and nothing else 
4. the barcode is followed by the fixed downstream sequence `GATCA` 
5. reads may arrive in lower case, and barcodes are reported in upper case 

Anything that fails one of these is not a valid barcode, and the parser should return `None` for it.

This list is the thing we are going to test against. 

#### Step 2: the parser we inherited, and a way to test it

In [ ]:
inherited_pattern = re.compile(r"CTAGC(?P<barcode>[A-Z]{6})GATCA")


def parse_barcode_v1(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = inherited_pattern.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")

A __test case__ is one input paired with the answer we expect. The `assert` statement is the
simplest way to check one: `assert <something true>` does nothing, and `assert <something false>`
stops with an error (as we saw in the warm-up).

Stopping at the first failure is unfriendly when you are debugging, so here is a small helper that
runs __every__ case and hands back all of the failures at once. You will reuse this.

In [ ]:
def check(parse_function, cases):
    """Run a parsing function against test cases and report any failures.

    Args:
        parse_function: Function that takes one sequence and returns a barcode
            string, or None if the sequence does not match.
        cases: List of (sequence, expected_result) tuples.

    Returns:
        A list of the cases that failed, as (sequence, expected, actual) tuples.
    """
    failures = []
    for sequence, expected in cases:
        actual = parse_function(sequence)
        if actual != expected:
            failures.append((sequence, expected, actual))

    return failures

----
### __Practice 3a (in groups, ~10 minutes): turn the requirements into test cases__

__Your task:__ fill in `test_cases` below so that it tests R1-R5.

Work down the requirement list and, for each one, ask: __what input would violate this requirement,
and only this one?__ That input is a __negative__ case, and its expected answer is `None`. Negative
cases matter more than you would think - a pattern that is too permissive can *only* be caught by
an input that should be rejected.

Include at least one __positive__ case too (a good read, and the barcode it should give back).

Two are filled in to show the format. Then run `check` and see how the inherited parser does.

In [ ]:
# each entry is (input sequence, expected barcode)
test_cases = [
    ("CTAGCatcgatGATCA", "ATCGAT"),  # positive: a good read, lower case
    ("CCAGCatagcaGATCA", None),  # violates R1: upstream sequence is CCAGC, not CTAGC
    # YOUR CASES HERE
]

check(parse_barcode_v1, test_cases)

#### Step 3: now run it on a list of reads

__Are any of the results unexpected?__

In [ ]:
seqs = [
    "CTAGCatcgatGATCA",
    "CTAGCgaccatGATCA",
    "CTAGCATYGATGATCA",
    "GGGGGCTAGCATCGATGATCATTTT",
    "CTAGCatcgatGATCA",
    "CTAGCtacagGATCA",
    "CCAGCatagcaGATCA",
]

for seq in seqs:
    print(f"{seq:<28} -> {parse_barcode_v1(seq)}")





----
### __Practice 3b (in groups, ~10 minutes): what went wrong, and what did our tests miss?__

Two of those reads came back with a "barcode" that is not a barcode. Find them, and for each one
work out:

1. __which requirement__ does that read violate?
2. __why didn't our test set catch it?__ Hint: a single requirement can be violated in more than one way...

Our test set was not *wrong*. It was __incomplete__, and the "real" data found holes.

__Your task:__ add the missing cases to `tricky_cases` below, then re-run `check`. You are trying to
make it fail.

In [ ]:
# add the cases our requirement list missed, then run check() again
tricky_cases = [
    # (sequence, expected)
]

check(parse_barcode_v1, test_cases + tricky_cases)

#### Step 4: fix the pattern so that it passes all of the tests!

In [ ]:
fixed_pattern = re.compile(r"")  # <-- your fixed pattern here


def parse_barcode_v2(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = fixed_pattern.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")


check(parse_barcode_v2, test_cases + tricky_cases)

Re-running the parser over the sequencing reads now gives `None` for everything that is not our
molecule - and the three real barcodes survive.

In [ ]:
for seq in seqs:
    print(f"{seq:<28} -> {parse_barcode_v2(seq)}")

----
# __Part 5: Making the parser reusable__

Our pattern has `CTAGC`, `6`, and `GATCA` baked into it. The next experiment will use a different
construct, and we do not want to rewrite - and re-test - the whole thing then.

### __Practice 4 (in groups, ~15 minutes)__

__Part (a):__ finish `count_barcodes` below. It should take the design as __arguments with default
values__, and return a dictionary of barcode counts. Your function needs to:

1. build the search pattern __from the arguments__, so the same function works for any design
   (`bclen` is an integer, so you will need `str(bclen)` to paste it into the pattern)
2. convert each sequence to uppercase
3. search it, and skip the sequence if there is no match
4. if it matches, extract the barcode and add it to a dictionary of counts

__Part (b):__ before you trust it, __write tests for it__ in the cell after. We know what the right
answer is for `seqs`: `ATCGAT` appears twice, `GACCAT` appears once, and three sequences should be
rejected. Write `assert` statements for those. Then write one more test that checks the
__arguments actually do something__ - for instance, parsing these same sequences with a different
expected `upstream` sequence should find nothing at all.

In [ ]:
def count_barcodes(seqs, bclen=6, upstream="CTAGC", downstream="GATCA"):
    """Parse and count barcodes.

    Args:
        seqs: List of DNA sequences.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode.
        downstream: Sequence expected immediately downstream of the barcode.

    Returns:
        A dict keyed by each valid barcode, where the value is the number of
        times that barcode is observed.

    Note:
        The function is **not** case-sensitive, and all barcodes are reported
        in upper case. The construct must start at the beginning of the sequence.
    """

    # YOUR CODE HERE


print(count_barcodes(seqs))

In [ ]:
# Part (b): write your tests here
counts = count_barcodes(seqs)

# YOUR ASSERTS HERE


print("all checks passed")

----
# __Wrapping up__

1. Data arrives as semi-structured text, and we often want to pull some sort of structure out of it.
2. You can write pattern search algorithms by hand; regular expressions (and the `re`) provide a more flexible and compact way to do that.
3. Being able to __read__ a pattern and translate it into English is a helpful first step
4. Wrong patterns can fail silently, so write test cases (including __negative__ ones)!

#### On Thursday (lecture 09)

We will use the `biopython` module to read a real FASTQ file from an influenza sequencing run